# MM-RAG — Reranker LoRA training (Week 3, Days 4-5)

Fine-tunes a cross-encoder reranker with a LoRA adapter on mined hard-negative
pairs. **Runtime > Change runtime type > T4 GPU** before running.

Day 4 = smoke test (SMOKE=True). Day 5 = full run (SMOKE=False).


## 1. Install


In [ ]:
!pip -q install transformers peft accelerate


## 2. Upload `train_pairs.jsonl` (from data/processed/)


In [ ]:
from google.colab import files
up = files.upload()   # select train_pairs.jsonl


## 3. Load + split


In [ ]:
import json, random
rows = [json.loads(l) for l in open("train_pairs.jsonl")]
random.seed(0); random.shuffle(rows)
val, train = rows[:300], rows[300:]
print("train", len(train), "val", len(val),
      "| positives", sum(r["label"] for r in rows), "of", len(rows))


## 4. Model + LoRA adapter


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

BASE = "cross-encoder/ms-marco-MiniLM-L-6-v2"
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=1)

lora = LoraConfig(task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16,
                  lora_dropout=0.1, target_modules=["query", "value"])
model = get_peft_model(model, lora)
model.print_trainable_parameters()   # should be a tiny % of total

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("device:", device)


## 5. Dataset


In [ ]:
from torch.utils.data import Dataset, DataLoader

class PairDS(Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]; return r["query"], r["passage"], float(r["label"])

def collate(batch):
    q, p, y = zip(*batch)
    enc = tok(list(q), list(p), truncation=True, max_length=256,
              padding=True, return_tensors="pt")
    enc["labels"] = torch.tensor(y).unsqueeze(1)
    return enc

SMOKE = True                     # Day 4: True (fast). Day 5: set False.
tr = train[:500] if SMOKE else train
tl = DataLoader(PairDS(tr), batch_size=16, shuffle=True, collate_fn=collate)
vl = DataLoader(PairDS(val), batch_size=32, collate_fn=collate)
print("training on", len(tr), "rows")


## 6. Train + evaluate


In [ ]:
import torch.nn as nn
opt = torch.optim.AdamW(model.parameters(), lr=2e-4)
lossf = nn.BCEWithLogitsLoss()

def val_acc():
    model.eval(); corr = tot = 0
    with torch.no_grad():
        for enc in vl:
            labels = enc.pop("labels").to(device)
            enc = {k: v.to(device) for k, v in enc.items()}
            pred = (torch.sigmoid(model(**enc).logits) > 0.5).float()
            corr += (pred == labels).sum().item(); tot += labels.numel()
    return corr / tot

print("val acc before:", round(val_acc(), 3))
EPOCHS = 1 if SMOKE else 3
for ep in range(EPOCHS):
    model.train()
    for i, enc in enumerate(tl):
        labels = enc.pop("labels").to(device)
        enc = {k: v.to(device) for k, v in enc.items()}
        loss = lossf(model(**enc).logits, labels)
        loss.backward(); opt.step(); opt.zero_grad()
        if i % 10 == 0: print(f"ep{ep} step{i} loss {loss.item():.4f}")
    print(f"ep{ep} val acc:", round(val_acc(), 3))


## 7. Save + download adapter


In [ ]:
name = "reranker_lora_smoke" if SMOKE else "reranker_lora_v1"
model.save_pretrained(name)
!zip -qr {name}.zip {name}
from google.colab import files
files.download(f"{name}.zip")
print("saved", name)
